<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session05_evaluating_llm_applications.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۵: ارزیابی برنامه‌های مبتنی بر LLM

### اهداف این جلسه

- فهم اینکه چرا «به نظرم بهتر شد» کافی نیست و ارزیابی چگونه توسعه را هدایت می‌کند
- ساخت مجموعه آزمون و محاسبه دقت، Precision، Recall، F1 و ماتریس درهم‌ریختگی
- مقایسه منصفانه پرامپت‌ها از نظر کیفیت، تأخیر و تعداد توکن
- تخمین عدم قطعیت نتایج با بازه اطمینان Bootstrap
- ارزیابی پاسخ‌های آزاد با LLM-as-a-Judge و شناخت سوگیری‌های داور

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. چرا ارزیابی؟

خروجی مدل‌های زبانی تصادفی و حساس به جزئیات پرامپت است. تغییری که یک نمونه را درست می‌کند ممکن است ده نمونه دیگر را خراب کند. بدون اندازه‌گیری، تصمیم‌ها بر اساس چند مثال دلبخواه گرفته می‌شوند. به این کار «Vibe Check» می‌گویند.

**توسعه ارزیابی‌محور (Evaluation-Driven Development)** همان نقشی را دارد که تست در برنامه‌نویسی معمولی دارد:

1. یک مجموعه آزمون ثابت می‌سازیم.
2. هر نسخه پرامپت، مدل یا معماری را روی آن اجرا می‌کنیم.
3. نتیجه را با عدد مقایسه می‌کنیم و تصمیم می‌گیریم.
4. هر خطای جدیدی که در عمل دیدیم، به مجموعه آزمون اضافه می‌شود.

| نوع خروجی | نمونه | روش ارزیابی |
|---|---|---|
| برچسب یا مقدار مشخص | طبقه‌بندی احساس، استخراج قیمت | مقایسه مستقیم با پاسخ درست: دقت، F1 |
| ساختاریافته | JSON استخراج‌شده | اعتبار Schema و دقت هر فیلد |
| متن آزاد | پاسخ پشتیبانی، خلاصه | داور انسانی یا LLM-as-a-Judge با معیار مشخص |
| رفتار چندمرحله‌ای | ایجنت | موفقیت در انجام کار، تعداد گام، هزینه |

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## راه‌اندازی مدل زبانی

همه نوت‌بوک‌های این دوره از طریق API سازگار با OpenAI با مدل حرف می‌زنند. تابع `setup_llm` از فایل `llm_utils.py` دو حالت دارد:

- `setup_llm("ollama")`: مدل متن‌باز `qwen2.5:7b` را با Ollama روی همین ماشین اجرا می‌کند. اجرای اول چند دقیقه برای نصب و دانلود مدل طول می‌کشد. در Colab حتماً **T4 GPU** را فعال کنید.
- `setup_llm("api")`: به هر سرویس سازگار با OpenAI وصل می‌شود. نشانی، کلید و نام مدل را در بخش **Secrets** کولب (آیکون کلید در نوار کناری) با نام‌های `LLM_BASE_URL`، `LLM_API_KEY` و `LLM_MODEL` ذخیره کنید.

جزئیات کار این تابع در جلسه ۱ توضیح داده شده است.

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. مجموعه آزمون

مجموعه آزمون خوب این ویژگی‌ها را دارد:

- **از داده واقعی** یا دست‌کم شبیه آن ساخته شده است.
- **همه حالت‌ها** را پوشش می‌دهد، به‌خصوص موارد دشوار و مرزی.
- **پاسخ درست** هر نمونه با دقت تعیین شده و در صورت امکان دو نفر آن را تأیید کرده‌اند.
- **ثابت و نسخه‌دار** است. اگر عوض شود، نتایج قبلی قابل مقایسه نیستند.
- از داده‌ای که برای **ساختن** پرامپت استفاده شده، مثل مثال‌های Few-shot، **جداست**.

در عمل دو مجموعه نگه می‌داریم: **Dev** برای آزمایش و بهبود مداوم، و **Test** که فقط گاهی و برای گزارش نهایی اجرا می‌شود. اگر بارها روی Test بهینه کنیم، بیش‌برازش روی همان مجموعه رخ می‌دهد.

از داده نظرات جلسه قبل استفاده می‌کنیم.

</div>

In [ ]:
import time

import numpy as np
import pandas as pd
from llm_utils import download_data

reviews = pd.read_csv(download_data("reviews.csv"))
LABELS = ["positive", "negative", "neutral"]
print(len(reviews), "reviews:", reviews.label.value_counts().to_dict())

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. یک چارچوب ارزیابی ساده

هر «سیستم» یک تابع است که متن را می‌گیرد و برچسب برمی‌گرداند. تابع `run_eval` برای هر نمونه پیش‌بینی، زمان پاسخ و تعداد توکن را ثبت می‌کند. همه نتایج در یک DataFrame می‌روند تا تحلیل‌شان آسان باشد.

</div>

In [ ]:
def parse_label(text):
    text = (text or "").strip().lower()
    for label in LABELS:
        if label in text:
            return label
    return "unknown"


def call(messages):
    """One deterministic request; returns (text, latency_seconds, total_tokens)."""
    start = time.time()
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    latency = time.time() - start
    usage = response.usage
    return response.choices[0].message.content, latency, usage.total_tokens if usage else None


def run_eval(name, build_messages, data=reviews):
    rows = []
    for _, item in data.iterrows():
        answer, latency, tokens = call(build_messages(item.text))
        rows.append({"system": name, "id": item.review_id, "label": item.label,
                     "prediction": parse_label(answer), "latency": latency, "tokens": tokens})
    return pd.DataFrame(rows)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

سه نسخه پرامپت را مقایسه می‌کنیم: دستورالعمل انگلیسی Zero-shot، همان دستورالعمل به فارسی، و نسخه Few-shot.

</div>

In [ ]:
SYSTEM_EN = """You classify Persian customer reviews of an online electronics store.
Labels:
- positive: the customer is satisfied with the product, delivery or service.
- negative: the customer reports a problem or is dissatisfied.
- neutral: a factual statement or an opinion that is neither clearly positive nor negative.
Answer with exactly one word: positive, negative or neutral."""

SYSTEM_FA = """تو نظرات فارسی مشتریان یک فروشگاه اینترنتی لوازم الکترونیکی را دسته‌بندی می‌کنی.
برچسب‌ها:
- positive: مشتری از کالا، ارسال یا خدمات راضی است.
- negative: مشتری مشکلی گزارش می‌کند یا ناراضی است.
- neutral: بیان یک واقعیت یا نظری که به‌وضوح مثبت یا منفی نیست.
فقط یکی از سه کلمه positive، negative یا neutral را بنویس."""

EXAMPLES = [
    ("مانیتور بدون حتی یک پیکسل سوخته رسید، رنگ‌هاش هم عالیه.", "positive"),
    ("دو هفته از تاریخ تحویل گذشته و هنوز خبری از سفارشم نیست.", "negative"),
    ("این ماوس با باتری قلمی کار می‌کنه و باتریش داخل جعبه نیست.", "neutral"),
    ("کیبورد خوبیه ولی نور پس‌زمینه‌اش ضعیفه.", "neutral"),
]


def zero_shot_en(text):
    return [{"role": "system", "content": SYSTEM_EN}, {"role": "user", "content": f"Review: {text}"}]


def zero_shot_fa(text):
    return [{"role": "system", "content": SYSTEM_FA}, {"role": "user", "content": f"Review: {text}"}]


def few_shot_en(text):
    messages = [{"role": "system", "content": SYSTEM_EN}]
    for example, label in EXAMPLES:
        messages += [{"role": "user", "content": f"Review: {example}"},
                     {"role": "assistant", "content": label}]
    return messages + [{"role": "user", "content": f"Review: {text}"}]


results = pd.concat([
    run_eval("zero-shot EN", zero_shot_en),
    run_eval("zero-shot FA", zero_shot_fa),
    run_eval("few-shot EN", few_shot_en),
], ignore_index=True)
results.head()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. معیارهای طبقه‌بندی

**دقت (Accuracy)** ساده‌ترین معیار است، ولی کافی نیست. وقتی کلاس‌ها نامتوازن‌اند، یا اشتباه در یک کلاس مهم‌تر از بقیه است، به معیارهای هر کلاس نیاز داریم:

- **Precision:** از نمونه‌هایی که مدل گفت «منفی»، چند درصد واقعاً منفی بودند؟
- **Recall:** از نظرات منفی واقعی، مدل چند درصد را پیدا کرد؟
- **F1:** میانگین هارمونیک این دو.
- **Macro-F1:** میانگین F1 همه کلاس‌ها با وزن برابر.

برای فروشگاه، Recall کلاس منفی معمولاً مهم‌ترین عدد است: هر نظر منفی که پیدا نشود، یعنی مشتری ناراضی که کسی به او رسیدگی نکرده است.

</div>

In [ ]:
from sklearn.metrics import classification_report, f1_score


def summarize(df):
    return pd.Series({
        "accuracy": (df.prediction == df.label).mean(),
        "macro_f1": f1_score(df.label, df.prediction, labels=LABELS, average="macro", zero_division=0),
        "negative_recall": ((df.prediction == "negative") & (df.label == "negative")).sum()
                           / (df.label == "negative").sum(),
        "unknown": (df.prediction == "unknown").mean(),
        "latency_s": df.latency.mean(),
        "tokens": df.tokens.mean(),
    })


summary = results.groupby("system").apply(summarize, include_groups=False)
summary.round(3)

In [ ]:
best = summary.macro_f1.idxmax()
print(f"best system: {best}\n")
print(classification_report(results.query("system == @best").label,
                            results.query("system == @best").prediction,
                            labels=LABELS, zero_division=0))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**ماتریس درهم‌ریختگی (Confusion Matrix)** نشان می‌دهد کدام کلاس‌ها با هم اشتباه گرفته می‌شوند:

</div>

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, df) in zip(axes, results.groupby("system")):
    ConfusionMatrixDisplay.from_predictions(df.label, df.prediction, labels=LABELS, ax=ax,
                                            colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

به ستون تعداد توکن هم دقت کنید. Few-shot دقیق‌تر است، ولی هر درخواست چند برابر توکن ورودی دارد. در یک سرویس پولی با میلیون‌ها درخواست، این تفاوت هزینه واقعی دارد. **انتخاب نهایی همیشه موازنه کیفیت، تأخیر و هزینه است.**

## ۵. این اختلاف واقعی است؟

مجموعه آزمون ما ۴۸ نمونه دارد. با این تعداد، هر نمونه حدود ۲ درصد دقت است. اگر یک پرامپت ۸۵٪ و دیگری ۸۱٪ دقت بگیرد، این اختلاف ممکن است فقط شانس باشد.

**Bootstrap** راه ساده‌ای برای سنجیدن عدم قطعیت است. هزار بار از نمونه‌ها با جایگذاری نمونه‌برداری می‌کنیم و هر بار دقت را حساب می‌کنیم. پراکندگی این هزار عدد، بازه اطمینان دقت را نشان می‌دهد.

</div>

In [ ]:
def bootstrap_ci(correct, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    correct = np.asarray(correct, dtype=float)
    samples = rng.choice(correct, size=(n_boot, len(correct)), replace=True).mean(axis=1)
    return np.percentile(samples, [2.5, 97.5])


for name, df in results.groupby("system"):
    correct = (df.prediction == df.label).values
    low, high = bootstrap_ci(correct)
    print(f"{name:14} accuracy={correct.mean():.2f}   95% CI=[{low:.2f}, {high:.2f}]")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

اگر بازه‌ها هم‌پوشانی زیادی دارند، نمی‌توان با اطمینان گفت کدام بهتر است. یک مقایسه دقیق‌تر، **مقایسه جفتی** است: فقط نمونه‌هایی را می‌شماریم که دو سیستم روی آن‌ها اختلاف دارند. هرچه مجموعه آزمون بزرگ‌تر باشد، اختلاف‌های کوچک‌تر قابل تشخیص می‌شوند. برای تصمیم‌های مهم، صدها نمونه لازم است.

</div>

In [ ]:
pivot = results.assign(correct=results.prediction == results.label).pivot(
    index="id", columns="system", values="correct")
a, b = "few-shot EN", "zero-shot EN"
only_a = (pivot[a] & ~pivot[b]).sum()
only_b = (pivot[b] & ~pivot[a]).sum()
print(f"{a} right and {b} wrong: {only_a}")
print(f"{b} right and {a} wrong: {only_b}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. ارزیابی پاسخ‌های آزاد: LLM-as-a-Judge

برای پاسخ‌های متنی آزاد، یک «پاسخ درست» یکتا وجود ندارد و مقایسه کلمه‌به‌کلمه بی‌معناست. داور انسانی بهترین است، ولی گران و کند است. راه رایج، استفاده از یک مدل زبانی به‌عنوان داور است: **LLM-as-a-Judge**.

کلید داوری خوب، **معیار (Rubric)** روشن است. داور باید بداند دقیقاً چه چیزی را بسنجد و هر نمره یعنی چه.

مثال: دو نسخه دستیار پشتیبانی نوین‌کالا را مقایسه می‌کنیم. هر دو سند مرتبط را می‌بینند. پرسش‌ها و پاسخ‌های مرجع از فایل `rag_eval.jsonl` می‌آیند. RAG کامل را در جلسه ۸ می‌سازیم؛ اینجا سند درست را دستی به مدل می‌دهیم.

</div>

In [ ]:
import json

from llm_utils import load_store_docs

docs = load_store_docs()
with open(download_data("rag_eval.jsonl"), encoding="utf-8") as f:
    qa = [json.loads(line) for line in f][:10]

ASSISTANTS = {
    "A_brief": "تو دستیار پشتیبانی فروشگاه نوین‌کالا هستی. فقط بر اساس سند داده‌شده و در حداکثر یک جمله پاسخ بده.",
    "B_helpful": ("تو دستیار پشتیبانی مؤدب فروشگاه نوین‌کالا هستی. فقط بر اساس سند داده‌شده پاسخ بده. "
                  "پاسخ را کامل ولی کوتاه بنویس و اگر شرط یا استثنایی وجود دارد، آن را هم بگو."),
}


def answer(system, item):
    context = docs[item["source"]]
    prompt = f"<document>\n{context}\n</document>\n\nپرسش مشتری: {item['question']}"
    return chat(client, MODEL, prompt, system=system, temperature=0)


answers = {name: [answer(system, item) for item in qa] for name, system in ASSISTANTS.items()}
print(qa[0]["question"])
for name in answers:
    print(f"  {name}: {answers[name][0]}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### داوری با نمره

داور پاسخ را با **پاسخ مرجع** مقایسه می‌کند و نمره ۱ تا ۵ می‌دهد. از خروجی ساختاریافته استفاده می‌کنیم و از داور می‌خواهیم **اول دلیل را بنویسد و بعد نمره بدهد**. این همان استدلال مرحله‌به‌مرحله جلسه قبل است و نمره‌ها را پایدارتر می‌کند.

</div>

In [ ]:
from typing import Literal

from pydantic import BaseModel


class Grade(BaseModel):
    reasoning: str
    score: Literal[1, 2, 3, 4, 5]


JUDGE_RUBRIC = """You are grading answers of a Persian customer-support assistant.
Compare the ANSWER with the REFERENCE answer and grade factual correctness and completeness:
5 = all facts in the reference are present and correct, nothing wrong is added.
4 = correct, but a minor detail or condition is missing.
3 = partly correct; an important detail is missing or slightly wrong.
2 = mostly wrong or misleading, with some correct parts.
1 = wrong, irrelevant or a refusal although the reference has an answer.
Write a short reasoning first, then the score. Tone and length do not matter."""


def judge(question, reference, candidate):
    prompt = f"QUESTION: {question}\nREFERENCE: {reference}\nANSWER: {candidate}"
    response = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": JUDGE_RUBRIC}, {"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "Grade", "schema": Grade.model_json_schema()}},
    )
    return Grade.model_validate_json(response.choices[0].message.content)


grades = []
for i, item in enumerate(qa):
    for name in answers:
        g = judge(item["question"], item["answer"], answers[name][i])
        grades.append({"id": item["id"], "system": name, "score": g.score, "reasoning": g.reasoning})
grades = pd.DataFrame(grades)
grades.groupby("system").score.agg(["mean", "min"])

In [ ]:
pd.set_option("display.max_colwidth", 120)
grades.sort_values("score").head(6)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**داور را هم باید ارزیابی کرد.** چند نمونه را خودتان نمره بدهید و با نمره داور مقایسه کنید. اگر توافق کم است، معیار را روشن‌تر کنید یا مدل قوی‌تری را داور کنید. یک قاعده رایج: داور دست‌کم به قدرت مدلی باشد که ارزیابی می‌شود.

### داوری جفتی و سوگیری جایگاه

راه دیگر، **مقایسه جفتی (Pairwise)** است: دو پاسخ را به داور نشان می‌دهیم و می‌پرسیم کدام بهتر است. این روش برای سنجش تفاوت‌های ظریف، مثل لحن یا وضوح، حساس‌تر از نمره‌دادن است. ولی داورها **سوگیری جایگاه (Position Bias)** دارند: معمولاً پاسخ اول یا دوم را بیشتر ترجیح می‌دهند، فارغ از محتوا. برای آزمودن این سوگیری، هر مقایسه را دو بار و با جابه‌جا کردن ترتیب پاسخ‌ها انجام می‌دهیم.

</div>

In [ ]:
PAIRWISE = ("You compare two answers of a Persian customer-support assistant to the same question. "
            "Prefer the answer that is correct, complete and clear for a customer. "
            "Reply with exactly one character: 1 if the first answer is better, 2 if the second is better.")


def prefer(question, first, second):
    prompt = f"QUESTION: {question}\n\nANSWER 1: {first}\n\nANSWER 2: {second}"
    out = chat(client, MODEL, prompt, system=PAIRWISE, temperature=0, max_tokens=3).strip()
    return "1" if out.startswith("1") else "2"


records = []
for i, item in enumerate(qa):
    a, b = answers["A_brief"][i], answers["B_helpful"][i]
    order_ab = prefer(item["question"], a, b)          # A shown first
    order_ba = prefer(item["question"], b, a)          # B shown first
    winner_ab = "A" if order_ab == "1" else "B"
    winner_ba = "B" if order_ba == "1" else "A"
    records.append({"id": item["id"], "A first": winner_ab, "B first": winner_ba,
                    "consistent": winner_ab == winner_ba})
pairwise = pd.DataFrame(records)
print("consistent verdicts:", f"{pairwise.consistent.mean():.0%}")
print("wins when consistent:", pairwise.query("consistent")["A first"].value_counts().to_dict())
pairwise

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مقایسه‌هایی که با جابه‌جایی ترتیب نتیجه‌شان عوض می‌شود، قابل اعتماد نیستند و آن‌ها را «مساوی» حساب می‌کنیم. سوگیری‌های شناخته‌شده دیگر:

- **سوگیری طول:** داورها پاسخ‌های بلندتر را بیشتر ترجیح می‌دهند.
- **خودشیفتگی (Self-preference):** داور پاسخ‌هایی را که خودش یا مدل‌های هم‌خانواده‌اش نوشته‌اند بیشتر می‌پسندد.
- **اثر لحن:** پاسخ مطمئن و مؤدب، حتی اگر غلط باشد، نمره بالاتری می‌گیرد.

راه‌های کاهش: معیار صریح، پاسخ مرجع، اجرای دوطرفه، و بررسی دوره‌ای نمونه‌ای از داوری‌ها توسط انسان.

## ۷. ابزارهای ارزیابی

چارچوب ما چند تابع ساده بود و برای یادگیری کافی است. در پروژه‌های بزرگ‌تر از ابزارهای آماده استفاده می‌شود:

| ابزار | کاربرد |
|---|---|
| promptfoo | مقایسه پرامپت‌ها و مدل‌ها با فایل پیکربندی، مناسب CI |
| DeepEval | ارزیابی به سبک pytest با معیارهای آماده |
| Ragas | معیارهای ویژه RAG (جلسه ۹) |
| LangSmith و Langfuse | ثبت اجراها، ارزیابی و پایش در محیط عملیاتی |

مهم‌تر از ابزار، عادت است: **هیچ تغییری بدون اجرای ارزیابی پذیرفته نمی‌شود.** نتایج را با نسخه پرامپت و مدل ذخیره کنید تا تاریخچه تصمیم‌ها روشن بماند.

</div>

In [ ]:
summary.assign(model=MODEL, date=pd.Timestamp.now().date()).to_csv("eval_history.csv", mode="a")
print(open("eval_history.csv").read()[:500])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۸. جمع‌بندی

- ارزیابی برای LLM نقش تست را دارد: مجموعه ثابت، اجرای خودکار، تصمیم با عدد.
- مجموعه آزمون باید واقعی، متنوع، نسخه‌دار و جدا از داده ساخت پرامپت باشد.
- دقت کافی نیست. Precision، Recall، Macro-F1 و ماتریس درهم‌ریختگی تصویر کامل‌تری می‌دهند.
- با مجموعه کوچک، اختلاف‌های کوچک ممکن است شانسی باشند. بازه اطمینان را گزارش کنید.
- LLM-as-a-Judge با معیار روشن و پاسخ مرجع کار می‌کند، ولی سوگیری جایگاه، طول و لحن دارد و باید خودش هم ارزیابی شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. نسخه چهارمی از طبقه‌بند بسازید که Few-shot را با دستورالعمل فارسی ترکیب کند. آن را به جدول مقایسه اضافه کنید و بازه اطمینان را گزارش کنید.
۲. ده نظر جدید و دشوار بنویسید، مثلاً نظرات طعنه‌آمیز یا نظراتی که هم مثبت و هم منفی‌اند، و به مجموعه آزمون اضافه کنید. رتبه پرامپت‌ها تغییر می‌کند؟
۳. پنج پاسخ از جدول `grades` را خودتان با همان معیار نمره بدهید. توافق شما با داور چقدر است؟
۴. برای سنجش **سوگیری طول**، پاسخ `A_brief` را با افزودن یک جمله کلی و بی‌اطلاعات («امیدواریم از خرید خود لذت ببرید.») طولانی کنید و داوری جفتی را تکرار کنید. نتیجه چه تغییری کرد؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** نسخه چهارمی از طبقه‌بند بسازید که Few-shot را با دستورالعمل فارسی ترکیب کند. آن را به جدول مقایسه اضافه کنید و بازه اطمینان را گزارش کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** ده نظر جدید و دشوار بنویسید، مثلاً نظرات طعنه‌آمیز یا نظراتی که هم مثبت و هم منفی‌اند، و به مجموعه آزمون اضافه کنید. رتبه پرامپت‌ها تغییر می‌کند؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** پنج پاسخ از جدول `grades` را خودتان با همان معیار نمره بدهید. توافق شما با داور چقدر است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** برای سنجش **سوگیری طول**، پاسخ `A_brief` را با افزودن یک جمله کلی و بی‌اطلاعات («امیدواریم از خرید خود لذت ببرید.») طولانی کنید و داوری جفتی را تکرار کنید. نتیجه چه تغییری کرد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید